In [ ]:
%env WORKDIR=/tmp/vault
%env CERT_NAME=2026

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/tmp/vault/config.env")

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')
VAULT_CACERT = os.getenv('VAULT_CACERT')

## https://developer.hashicorp.com/vault/tutorials/secrets-management/pki-engine

![image.png](attachment:image.png)

In [ ]:
! curl -k --header "X-Vault-Token: $VAULT_TOKEN" --request POST --data '{"hmac":false}' $VAULT_ADDR/v1/sys/config/auditing/request-headers/my-header

## Step 1: Generate root CA

### Enable PKI engine==mount point

In [ ]:
! vault secrets enable pki 

In [ ]:
# Tune PKI to set max_tll
! vault secrets tune -max-lease-ttl=87600h pki

### Generate rootCA

In [ ]:
%%bash

vault write -field=certificate pki/root/generate/internal \
     common_name="example.com" alt_names="test.com" \
     issuer_name="root-$CERT_NAME" \
     ttl=87600h > ${WORKDIR}/root_$CERT_NAME_ca.crt

### # List CA information

In [ ]:
%%bash
# List CA information
export ISSUER=$(vault list -format=json pki/issuers/ | jq -r .[0])
echo $ISSUER

vault read pki/issuer/$ISSUER | tail -n 11

[PKI Role](https://developer.hashicorp.com/vault/api-docs/secret/pki#create-update-role) details

In [ ]:
%%bash
# Create a role that will allow for using certificates (in this case any name will be valid)
vault write pki/roles/$CERT_NAME-servers allow_any_name=true no_store=false

### Configure CA and CRL URLs

In [ ]:
%%bash
# Configure Vault cluster URLs
vault write pki/config/cluster \
   path=https://vault.vault.svc.cluster.local:8200/v1/pki \
   aia_path=https://vault.vault.svc.cluster.local:8200/v1/pki

In [ ]:
%%bash

vault write pki/config/urls \
   issuing_certificates={{cluster_aia_path}}/issuer/{{issuer_id}}/der \
   crl_distribution_points={{cluster_aia_path}}/issuer/{{issuer_id}}/crl/der \
   ocsp_servers={{cluster_path}}/ocsp \
   enable_templating=true

[OCSP](https://developer.hashicorp.com/vault/api-docs/secret/pki#ocsp-request)

## Step 2: Generate intermediate CA

### The intermediate CA is expressed a another PKI engine with a separate mount point

In [ ]:
! vault secrets enable -path=pki_int pki

In [ ]:
# the mount is configured with a max_tll
! vault secrets tune -max-lease-ttl=43800h pki_int

### Generate Intermediate CA whose CSR is going to be signed by the root CA at pki mount path

In [ ]:
%%bash

vault write -format=json pki_int/intermediate/generate/internal \
     common_name="example.com Intermediate Authority" \
     issuer_name="example-dot-com-intermediate" \
     | jq -r '.data.csr' > $WORKDIR/pki_intermediate.csr

### Send intermediateCA CSR for signing with CA mount point

In [ ]:
%%bash

vault write -format=json pki/root/sign-intermediate \
     issuer_ref="root-$CERT_NAME" \
     csr=@$WORKDIR/pki_intermediate.csr \
     format=pem_bundle ttl="43800h" \
     | jq -r '.data.certificate' > ${WORKDIR}/intermediate.cert.pem

### Import signed intermediate CA (`intermediate.cer.pem`) to its correspondant mount point

In [ ]:
# Import signed intermediate CA to its correspondant mount point
! vault write pki_int/intermediate/set-signed certificate=@${WORKDIR}/intermediate.cert.pem

### Configure CA and CRL URLs for intermediate CA

In [ ]:
%%bash
# Configure Vault cluster URLs
vault write pki_int/config/cluster \
   path=https://vault.vault.svc.cluster.local:8200/v1/pki_int \
   aia_path=https://vault.vault.svc.cluster.local:8200/v1/pki_int

In [ ]:
%%bash
vault write pki_int/config/urls \
   issuing_certificates={{cluster_aia_path}}/issuer/{{issuer_id}}/der \
   crl_distribution_points={{cluster_aia_path}}/issuer/{{issuer_id}}/crl/der \
   ocsp_servers={{cluster_path}}/ocsp \
   enable_templating=true

## Step 3: Create role -> https://developer.hashicorp.com/vault/tutorials/secrets-management/pki-engine#step-3-create-a-role

### Create role that allow certificates to be signed for domain `example.com` and `test.com` with intermediate CA (`pki_int` mount point)

In [ ]:
%%bash
# Create role that allow certificates to be signed for domain `example.com` and `test.com`
vault write pki_int/roles/example-dot-com \
     issuer_ref="$(vault read -field=default pki_int/config/issuers)" \
     allowed_domains="example.com","test.com" \
     allow_subdomains=true \
     allow_glob_domains=true \
     allow_wildcard_certificates=true \
     allow_ip_sans=true \
     allowed_uri_sans="*.example.com" \
     max_ttl="24h" \
     ttl="12h" \
     ext_key_usage="Client Auth"

## Step 4: Requests certificates

### Generate Certificates using Vault CLI

In [ ]:
! vault write pki_int/issue/example-dot-com common_name="*.test.com" ip_sans="8.8.8.9" \
uri_sans="otrauri.example.com,masuri.example.com" ttl="1m" 

### if you want to see details on the audit logs about the certificate information unhash

In [ ]:
%%bash

# Tune to unhash request and response values
vault secrets tune  \
     -max-lease-ttl="43800h"  -audit-non-hmac-request-keys="csr" -audit-non-hmac-request-keys="certificate" -audit-non-hmac-request-keys=issuer_ref -audit-non-hmac-request-keys="common_name" -audit-non-hmac-request-keys=alt_names -audit-non-hmac-request-keys=other_sans  \
     -audit-non-hmac-request-keys="ip_sans" -audit-non-hmac-request-keys=uri_sans  -audit-non-hmac-request-keys=ttl  -audit-non-hmac-request-keys=not_after  \
     -audit-non-hmac-request-keys=serial_number -audit-non-hmac-request-keys=key_type -audit-non-hmac-request-keys=private_key_format \
     -audit-non-hmac-request-keys=ou -audit-non-hmac-request-keys=organization -audit-non-hmac-request-keys=country \
     -audit-non-hmac-request-keys=locality -audit-non-hmac-request-keys=province -audit-non-hmac-request-keys=street_address \
     -audit-non-hmac-request-keys=postal_code -audit-non-hmac-request-keys=permitted_dns_domains -audit-non-hmac-request-keys=policy_identitiers \
     -audit-non-hmac-request-keys=ext_key_usage_oids -audit-non-hmac-response-keys=certificate -audit-non-hmac-response-keys=issuing_ca -audit-non-hmac-response-keys=error  \
     -audit-non-hmac-response-keys=serial_number -audit-non-hmac-response-keys=ca_chain -audit-non-hmac-response-keys=private_key_type -audit-non-hmac-response-keys=expiration pki_int

### Generate Certificates using the API

In [ ]:
%%bash

curl -k --header "X-Vault-Token: $VAULT_TOKEN" --request PUT --silent --data '{"common_name": "hash123.example.com", "ttl": "2h"}' $VAULT_ADDR/v1/pki_int/issue/example-dot-com | jq

In [ ]:
%%bash
export CERT_NAME=san.example.com

# Using CURL
curl -k --header "X-Vault-Token: $VAULT_TOKEN"\
    --request POST --silent \
    --data '{"common_name": "'"$CERT_NAME"'", "ttl": "1m"}' \
    $VAULT_ADDR/v1/pki_int/issue/example-dot-com > ${WORKDIR}/mycert.json

cat ${WORKDIR}/mycert.json | jq -r .data.certificate | openssl x509 -in /dev/stdin -text -noout

In [ ]:
%%bash
export CERT_NAME="test107.test.com"

#Using CURL
curl -k --header "X-Vault-Token: $VAULT_TOKEN"\
    --request POST --silent\
    --data '{"common_name": "'"$CERT_NAME"'", "ttl": "1m"}' \
    $VAULT_ADDR/v1/pki_int/issue/example-dot-com > ${WORKDIR}/mycert.json

cat ${WORKDIR}/mycert.json | jq -r .data.certificate > ${WORKDIR}/mycert_leaf.pem
cat ${WORKDIR}/mycert.json | jq -r .data.private_key > ${WORKDIR}/mycert_key.pem
openssl x509 -in  ${WORKDIR}/mycert_leaf.pem -text -noout

## Read Certificates

### List Certificates

In [ ]:
%%bash


curl -k \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request LIST --silent\
    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys

### Read Details about a certificate based on serial number
> #### Note private key can just be retrieved at creation time

In [ ]:
%%bash
export SERIAL=$(curl -k --silent\
                    --header "X-Vault-Token: $VAULT_TOKEN"\
                    --request LIST \
                    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys[0])
    

curl -k --silent\
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request GET \
    $VAULT_ADDR/v1/pki_int/cert/$SERIAL | jq -r .data.certificate > ${WORKDIR}/temp.pem
    
openssl x509 -in  ${WORKDIR}/temp.pem -text -noout 

### Revoke certificate based on serial number

In [ ]:
%%bash

# https://developer.hashicorp.com/vault/api-docs/secret/pki#revoke-certificate
export SERIAL=$(curl -k --silent\
                    --header "X-Vault-Token: $VAULT_TOKEN" \
                    --request LIST \
                    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys[3])
    
curl -k --silent --header "X-Vault-Token: $VAULT_TOKEN" \
    --request POST \
    --data '{"serial_number": "'"$SERIAL"'"}' \
    $VAULT_ADDR/v1/pki_int/revoke | jq

In [ ]:
%%bash
export SERIAL2=$(curl -k --silent\
                    --header "X-Vault-Token: $VAULT_TOKEN" \
                    --request LIST \
                    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys[1])

vault write pki_int/revoke serial_number=$SERIAL2

### Let's check the CRL

In [ ]:
%%bash
curl -k --silent  $VAULT_ADDR/v1/pki_int/crl -o $WORKDIR/out.crl
openssl crl -inform DER -text -noout -in $WORKDIR/out.crl

### Vault does not remove the certificate from its list of `cert store` until a tidy operation is run

In [ ]:
%%bash
echo "Todos los certificados"
curl -k \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request LIST --silent\
    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys

### Two types of `tidy` operation one-of or automatic

In [ ]:
%%bash
# https://developer.hashicorp.com/vault/api-docs/secret/pki#tidy
vault write pki_int/tidy tidy_cert_store=true tidy_revoked_certs=true safety_buffer=1m

# Auto Tidy
vault write pki_int/config/auto-tidy tidy_cert_store=true tidy_revoked_certs=true safety_buffer=60m

### List Certificates

In [ ]:
%%bash

echo "Todos los certificados"
curl -k \
    --header "X-Vault-Token: $VAULT_TOKEN" \
    --request LIST --silent\
    $VAULT_ADDR/v1/pki_int/certs | jq -r .data.keys



## Clean UP

In [ ]:
! vault audit enable file file_path=stdout

# EST Testing

[Enrollment over Secure Transport (EST)](https://developer.hashicorp.com/vault/docs/secrets/pki/est) (RFC 7030) is a Vault **Enterprise** PKI protocol that lets devices obtain CA certificates and enroll / re-enroll client certificates.

This section stands up a **dedicated intermediate mount** (`pki_est`) and wires:

1. **HTTP Basic** authentication (`userpass`, batch tokens)
2. **TLS client-certificate** authentication (`cert`, batch tokens)
3. **`simpleenroll`** and **`simplereenroll`** on `/.well-known/est/`

EST clients call `https://<vault>/.well-known/est/{cacerts,simpleenroll,simplereenroll}` (not `/v1/...`). Vault authenticates each request by delegating to those auth mounts, then returns a base64-encoded PKCS#7 (CMS certs-only) payload.

> Requires Vault Enterprise. Only one PKI mount in the cluster can set `default_mount=true`.

## Step 1: Dedicated intermediate CA (`pki_est`)

Reuse the existing root at `pki/` to sign a new intermediate used **only** for EST issuance. Keep it separate from `pki_int` so EST policy, roles, and auth accessors do not collide with the earlier demo.

### Enable the EST PKI mount and raise `max_lease_ttl`

In [ ]:
%%bash
mkdir -p ${WORKDIR}/est

# Vault EST returns MIME base64 PKCS#7 with CRLF (RFC 7030).
# macOS LibreSSL `openssl base64 -d -A` yields an empty decode on that format;
# wrapping as PEM PKCS#7 is portable.
cat > ${WORKDIR}/est/p7_to_pem.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
{
  printf '%s\n' '-----BEGIN PKCS7-----'
  tr -d '\r' < "$1"
  printf '%s\n' '-----END PKCS7-----'
} | openssl pkcs7 -inform PEM -print_certs -out "$2"
EOF
chmod +x ${WORKDIR}/est/p7_to_pem.sh

vault secrets enable -path=pki_est pki 2>/dev/null || echo "pki_est/ already enabled"
vault secrets tune -max-lease-ttl=43800h pki_est

### Generate the intermediate CSR (key stays inside Vault)

In [ ]:
%%bash
vault write -format=json pki_est/intermediate/generate/internal \
     common_name="example.com EST Intermediate Authority" \
     issuer_name="est-intermediate" \
     | jq -r '.data.csr' > ${WORKDIR}/est/pki_est_intermediate.csr

openssl req -in ${WORKDIR}/est/pki_est_intermediate.csr -noout -subject

### Sign the EST intermediate with the root CA at `pki/`

In [ ]:
%%bash
vault write -format=json pki/root/sign-intermediate \
     issuer_ref="root-$CERT_NAME" \
     csr=@${WORKDIR}/est/pki_est_intermediate.csr \
     format=pem_bundle ttl="43800h" \
     | jq -r '.data.certificate' > ${WORKDIR}/est/pki_est_intermediate.cert.pem

openssl x509 -in ${WORKDIR}/est/pki_est_intermediate.cert.pem -noout -subject -issuer

### Import the signed intermediate into `pki_est`

In [ ]:
# Import signed EST intermediate CA
! vault write pki_est/intermediate/set-signed certificate=@${WORKDIR}/est/pki_est_intermediate.cert.pem

### Configure cluster and AIA URLs for the EST intermediate

In [ ]:
%%bash
vault write pki_est/config/cluster \
   path=https://vault.vault.svc.cluster.local:8200/v1/pki_est \
   aia_path=https://vault.vault.svc.cluster.local:8200/v1/pki_est

vault write pki_est/config/urls \
   issuing_certificates={{cluster_aia_path}}/issuer/{{issuer_id}}/der \
   crl_distribution_points={{cluster_aia_path}}/issuer/{{issuer_id}}/crl/der \
   ocsp_servers={{cluster_path}}/ocsp \
   enable_templating=true

vault read pki_est/config/cluster
vault read -format=json pki_est/cert/ca | jq -r .data.certificate > ${WORKDIR}/est/pki_est_ca.pem
vault read -format=json pki/cert/ca | jq -r .data.certificate > ${WORKDIR}/est/root_ca.pem
cat ${WORKDIR}/est/pki_est_ca.pem ${WORKDIR}/est/root_ca.pem > ${WORKDIR}/est/ca_chain.pem
echo "EST intermediate CA subject:"
openssl x509 -in ${WORKDIR}/est/pki_est_ca.pem -noout -subject

## Step 2: EST issuance role

Role-based path policy (`role:est-clients`) restricts CNs/SANs that EST will sign. `use_csr_common_name` / `use_csr_sans` must be true so the PKCS#10 request drives identity. `Client Auth` EKU is required so issued certs can later authenticate via the `cert` method.

In [ ]:
%%bash
vault write pki_est/roles/est-clients \
     issuer_ref="$(vault read -field=default pki_est/config/issuers)" \
     allowed_domains="example.com,est.example.com" \
     allow_subdomains=true \
     allow_bare_domains=true \
     allow_glob_domains=true \
     allow_ip_sans=true \
     client_flag=true \
     server_flag=false \
     key_type=rsa \
     key_bits=2048 \
     max_ttl="24h" \
     ttl="12h" \
     not_before_duration="0s" \
     require_cn=true \
     use_csr_common_name=true \
     use_csr_sans=true \
     no_store=false \
     ext_key_usage="Client Auth"

vault read pki_est/roles/est-clients

## Step 3: EST authentication (HTTP Basic + TLS)

Vault EST does **not** consume `X-Vault-Token`. It delegates to dedicated auth mounts in the same namespace:

| EST credential | Vault auth mount | Token type |
| --- | --- | --- |
| HTTP Basic (`Authorization: Basic ...`) | `est-userpass` (`userpass`) | **batch** |
| TLS client certificate | `est-cert` (`cert`) | **batch** |

Batch tokens are mandatory: every EST request is authenticated, and service tokens would leak leases. ACL paths must match the **internal redirected** PKI path (`pki_est/roles/est-clients/est/...`), not `/.well-known/est/`.

### Policy for role-based EST enroll / re-enroll

In [ ]:
%%bash
vault policy write est-enroll - <<'EOF'
path "pki_est/roles/est-clients/est/simpleenroll" {
  capabilities = ["create", "update"]
}
path "pki_est/roles/est-clients/est/simplereenroll" {
  capabilities = ["create", "update"]
}
EOF

vault policy read est-enroll

### HTTP Basic: dedicated `userpass` mount (`est-userpass`)

In [ ]:
%%bash
vault auth enable -path=est-userpass userpass 2>/dev/null || echo "est-userpass/ already enabled"
vault auth tune -token-type=batch est-userpass

vault write auth/est-userpass/users/estuser \
     password="estpass" \
     token_policies="est-enroll" \
     token_type="batch" \
     token_ttl="5m"

echo "userpass accessor: $(vault read -field=accessor sys/auth/est-userpass)"
vault read sys/auth/est-userpass

### TLS: dedicated `cert` mount (`est-cert`)

Trust the EST intermediate CA so any leaf it issues (matching `*.est.example.com`) can authenticate. `cert_role=est-clients` is later passed from the EST config as the `name` parameter on cert login.

In [ ]:
%%bash
vault auth enable -path=est-cert cert 2>/dev/null || echo "est-cert/ already enabled"
vault auth tune -token-type=batch est-cert

vault write auth/est-cert/certs/est-clients \
     display_name="est-tls" \
     policies="est-enroll" \
     certificate=@${WORKDIR}/est/pki_est_ca.pem \
     allowed_common_names="*.est.example.com" \
     token_policies="est-enroll" \
     token_type="batch" \
     token_ttl="5m"

echo "cert accessor: $(vault read -field=accessor sys/auth/est-cert)"
vault read auth/est-cert/certs/est-clients

## Step 4: Tune the PKI mount and enable EST

1. Allow EST response headers (`Content-Transfer-Encoding`, `Content-Length`, `WWW-Authenticate`)
2. Register both auth accessors as `delegated-auth-accessors`
3. Enable EST, register the default `/.well-known/est/` prefix, and bind authenticators

See the [EST configuration API](https://developer.hashicorp.com/vault/api-docs/secret/pki/issuance#set-est-configuration).

In [ ]:
%%bash
USERPASS_ACCESSOR=$(vault read -field=accessor sys/auth/est-userpass)
CERT_ACCESSOR=$(vault read -field=accessor sys/auth/est-cert)

echo "USERPASS_ACCESSOR=${USERPASS_ACCESSOR}"
echo "CERT_ACCESSOR=${CERT_ACCESSOR}"

vault secrets tune \
  -allowed-response-headers="Content-Transfer-Encoding" \
  -allowed-response-headers="Content-Length" \
  -allowed-response-headers="WWW-Authenticate" \
  -delegated-auth-accessors="${USERPASS_ACCESSOR}" \
  -delegated-auth-accessors="${CERT_ACCESSOR}" \
  pki_est

vault write pki_est/config/est - <<EOF
{
  "enabled": true,
  "default_mount": true,
  "default_path_policy": "role:est-clients",
  "label_to_path_policy": {
    "est-clients": "role:est-clients"
  },
  "authenticators": {
    "cert": {
      "accessor": "${CERT_ACCESSOR}",
      "cert_role": "est-clients"
    },
    "userpass": {
      "accessor": "${USERPASS_ACCESSOR}"
    }
  },
  "enable_sentinel_parsing": true,
  "audit_fields": ["common_name", "alt_names", "ip_sans", "uri_sans", "csr"]
}
EOF

### Verify EST configuration (enabled, default mount, both authenticators)

In [ ]:
%%bash
vault read -format=json pki_est/config/est | jq .

echo
echo "=== delegated auth accessors + allowed response headers ==="
vault read -format=json sys/mounts/pki_est | jq '{
  delegated_auth_accessors: .data.config.delegated_auth_accessors,
  allowed_response_headers: .data.config.allowed_response_headers
}'

## Step 5: Verify `cacerts` (unauthenticated)

`GET /cacerts` must work **without** credentials and return the CA chain as PKCS#7. Exercise both the RFC path and the role-qualified API path.

Vault sends MIME base64 with CRLF (76-column). On macOS LibreSSL, `openssl base64 -d -A` produces an empty decode — wrap the body as PEM PKCS#7 (`p7_to_pem.sh`) instead.

In [ ]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"

cat > ${WORKDIR}/est/p7_to_pem.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail
{
  printf '%s\n' '-----BEGIN PKCS7-----'
  tr -d '\r' < "$1"
  printf '%s\n' '-----END PKCS7-----'
} | openssl pkcs7 -inform PEM -print_certs -out "$2"
EOF
chmod +x ${WORKDIR}/est/p7_to_pem.sh

echo "=== GET /.well-known/est/cacerts ==="
curl -k -sS -D ${WORKDIR}/est/cacerts.hdr -o ${WORKDIR}/est/cacerts.p7 \
  "${EST_BASE}/cacerts"
echo "--- response headers ---"
grep -iE 'HTTP/|content-type|content-transfer-encoding|content-length' ${WORKDIR}/est/cacerts.hdr

echo
echo "=== CA certificates from PKCS#7 ==="
# MIME base64 + CRLF: do not use `openssl base64 -d -A` on macOS LibreSSL
${WORKDIR}/est/p7_to_pem.sh ${WORKDIR}/est/cacerts.p7 ${WORKDIR}/est/cacerts.pem
grep -E 'subject=|issuer=' ${WORKDIR}/est/cacerts.pem || true
echo
openssl x509 -in ${WORKDIR}/est/cacerts.pem -noout -subject -issuer -dates

echo
echo "=== GET /v1/pki_est/roles/est-clients/est/cacerts ==="
curl -k -sS -D ${WORKDIR}/est/cacerts_api.hdr -o ${WORKDIR}/est/cacerts_api.p7 \
  "${VAULT_ADDR}/v1/pki_est/roles/est-clients/est/cacerts"
grep -iE 'HTTP/|content-type|content-transfer-encoding' ${WORKDIR}/est/cacerts_api.hdr
echo "well-known bytes=$(wc -c < ${WORKDIR}/est/cacerts.p7) api bytes=$(wc -c < ${WORKDIR}/est/cacerts_api.p7)" 

## Step 6: Enrollment and re-enrollment with HTTP Basic

RFC 7030: POST a base64 DER PKCS#10 (`Content-Type: application/pkcs10`) to `simpleenroll` / `simplereenroll`. Vault prefers HTTP Basic over a TLS client cert when both are present for **authentication**, but `/simplereenroll` still needs the existing leaf in the TLS handshake.

Credentials: `estuser` / `estpass`.

### Basic auth — `simpleenroll`

In [ ]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"
CN="device1.est.example.com"

openssl req -new -newkey rsa:2048 -nodes \
  -keyout ${WORKDIR}/est/device1.key \
  -out ${WORKDIR}/est/device1.csr \
  -subj "/CN=${CN}" 2>/dev/null
openssl req -in ${WORKDIR}/est/device1.csr -outform DER \
  | openssl base64 -e > ${WORKDIR}/est/device1.p10

echo "=== CSR ==="
openssl req -in ${WORKDIR}/est/device1.csr -noout -subject

echo
echo "=== POST /.well-known/est/simpleenroll (HTTP Basic) ==="
curl -k -sS -D ${WORKDIR}/est/enroll_basic.hdr \
  -o ${WORKDIR}/est/device1.p7 \
  --user "estuser:estpass" \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/device1.p10 \
  "${EST_BASE}/simpleenroll"

echo "--- response headers ---"
grep -iE 'HTTP/|content-type|content-transfer-encoding|www-authenticate' ${WORKDIR}/est/enroll_basic.hdr
HTTP_CODE=$(awk 'NR==1 {print $2}' ${WORKDIR}/est/enroll_basic.hdr)
if [ "${HTTP_CODE}" != "200" ]; then
  echo "EST enroll failed HTTP ${HTTP_CODE}"
  cat ${WORKDIR}/est/device1.p7; echo
  exit 1
fi

${WORKDIR}/est/p7_to_pem.sh ${WORKDIR}/est/device1.p7 ${WORKDIR}/est/device1.pem
# First PEM block is the issued leaf
openssl x509 -in ${WORKDIR}/est/device1.pem -out ${WORKDIR}/est/device1_leaf.pem
cp ${WORKDIR}/est/device1.key ${WORKDIR}/est/device1_leaf.key

echo
echo "=== issued certificate ==="
openssl x509 -in ${WORKDIR}/est/device1_leaf.pem -noout -subject -issuer -serial -dates
openssl x509 -in ${WORKDIR}/est/device1_leaf.pem -text -noout | grep -E "Public Key Algorithm:|Signature Algorithm:|X509v3 Extended Key Usage:|TLS Web Client" -A1

echo
echo "=== openssl verify against EST CA chain ==="
openssl verify -CAfile ${WORKDIR}/est/ca_chain.pem ${WORKDIR}/est/device1_leaf.pem

echo
echo "serial stored for re-enroll comparison:"
openssl x509 -in ${WORKDIR}/est/device1_leaf.pem -noout -serial | tee ${WORKDIR}/est/device1.serial

### Basic-only `simplereenroll` — expected failure (no TLS client cert)

[Vault EST docs](https://developer.hashicorp.com/vault/docs/secrets/pki/est) say HTTP Basic is **preferred over TLS client certs for authentication**. That is true for the delegated `userpass` mount (`simpleenroll` with Basic returns 200).

`/simplereenroll` still requires the **current leaf in the TLS handshake** to bind the identity being renewed ([RFC 7030 §4.2.2](https://www.rfc-editor.org/rfc/rfc7030#section-4.2.2)). Without `--cert/--key`, Vault returns **`400 no certificate found in TLS state`**.

RFC 7030 [§2.3](https://www.rfc-editor.org/rfc/rfc7030#section-2.3) allows a fallback (re-enroll with the same method as initial enrollment when the cert cannot be used for TLS client auth). **Vault does not implement that fallback.** Many network devices call `simplereenroll` with Basic only and cannot auto-renew.

The next cell is the working path: Basic (ACL) **plus** the existing client cert (identity).

In [ ]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"

echo "=== POST /.well-known/est/simplereenroll (HTTP Basic ONLY — no client cert) ==="
curl -k -sS -D ${WORKDIR}/est/reenroll_basic_only.hdr \
  -o ${WORKDIR}/est/reenroll_basic_only.body \
  --user "estuser:estpass" \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/device1.p10 \
  "${EST_BASE}/simplereenroll"

grep -iE 'HTTP/|content-type' ${WORKDIR}/est/reenroll_basic_only.hdr
echo "body: $(cat ${WORKDIR}/est/reenroll_basic_only.body)"

HTTP_CODE=$(awk 'NR==1 {print $2}' ${WORKDIR}/est/reenroll_basic_only.hdr)
if [ "${HTTP_CODE}" = "400" ] && grep -q "no certificate found in TLS state" ${WORKDIR}/est/reenroll_basic_only.body; then
  echo "EXPECTED: Basic authenticates, but simplereenroll still requires a TLS client certificate"
else
  echo "UNEXPECTED: wanted HTTP 400 + 'no certificate found in TLS state', got HTTP ${HTTP_CODE}"
  exit 1
fi

### Basic auth — `simplereenroll` (same identity, new key, new serial)

Vault EST requires the **currently issued client certificate in the TLS handshake** for `/simplereenroll` (without it: `no certificate found in TLS state`). HTTP Basic remains the Vault authenticator; `--cert/--key` prove possession of the existing leaf.

Vault then checks that the CSR matches that leaf **byte-for-byte**:

| Check | Failure message | Fix in the CSR |
| --- | --- | --- |
| Subject | `CSR Subject field does not match client certificate` | `string_mask = nombstr` so `CN` is `PrintableString` (LibreSSL `-subj` emits `UTF8String`) |
| SAN extension | `CSR SubjectAltName Extension does not match client certificate` | `subjectAltName = DNS:<CN>` **and** `openssl req -reqexts ext` (LibreSSL ignores `req_extensions` in the config otherwise) |

The CSR uses a **new key** (rekey). The cell verifies the renewed cert is bound to it, then rotates `device1_leaf.pem` / `device1.key` so the cell can be re-run.

In [ ]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"
CN="device1.est.example.com"
OLD_SERIAL=$(cat ${WORKDIR}/est/device1.serial)

# Rekey CSR: Subject and SAN must match the current leaf byte-for-byte.
#  - string_mask=nombstr -> CN as PrintableString (LibreSSL `-subj` emits UTF8String)
#  - -reqexts ext        -> LibreSSL omits req_extensions from the config unless passed on the CLI
#  - subjectAltName=DNS  -> Vault compares the SAN extension on reenroll
cat > ${WORKDIR}/est/reenroll_req.cnf <<EOF
[req]
distinguished_name = dn
req_extensions = ext
string_mask = nombstr
prompt = no
utf8 = no
[dn]
CN = ${CN}
[ext]
subjectAltName = DNS:${CN}
EOF
openssl req -new -newkey rsa:2048 -nodes \
  -keyout ${WORKDIR}/est/device1_re.key \
  -out ${WORKDIR}/est/device1_re.csr \
  -config ${WORKDIR}/est/reenroll_req.cnf \
  -reqexts ext
echo "=== renewal CSR (expect PRINTABLESTRING + DNS SAN) ==="
openssl asn1parse -in ${WORKDIR}/est/device1_re.csr | grep PRINTABLESTRING || true
openssl req -in ${WORKDIR}/est/device1_re.csr -noout -text | grep -A1 "Subject Alternative Name"
openssl req -in ${WORKDIR}/est/device1_re.csr -outform DER \
  | openssl base64 -e > ${WORKDIR}/est/device1_re.p10

echo "=== POST /.well-known/est/simplereenroll (HTTP Basic + current client cert) ==="
curl -k -sS -D ${WORKDIR}/est/reenroll_basic.hdr \
  -o ${WORKDIR}/est/device1_re.p7 \
  --user "estuser:estpass" \
  --cert ${WORKDIR}/est/device1_leaf.pem \
  --key ${WORKDIR}/est/device1.key \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/device1_re.p10 \
  "${EST_BASE}/simplereenroll"

grep -iE 'HTTP/|content-type|content-transfer-encoding' ${WORKDIR}/est/reenroll_basic.hdr
HTTP_CODE=$(awk 'NR==1 {print $2}' ${WORKDIR}/est/reenroll_basic.hdr)
if [ "${HTTP_CODE}" != "200" ]; then
  echo "EST reenroll failed HTTP ${HTTP_CODE}"
  cat ${WORKDIR}/est/device1_re.p7; echo
  exit 1
fi

${WORKDIR}/est/p7_to_pem.sh ${WORKDIR}/est/device1_re.p7 ${WORKDIR}/est/device1_re.pem
openssl x509 -in ${WORKDIR}/est/device1_re.pem -out ${WORKDIR}/est/device1_re_leaf.pem

NEW_SERIAL=$(openssl x509 -in ${WORKDIR}/est/device1_re_leaf.pem -noout -serial)
echo "old ${OLD_SERIAL}"
echo "new ${NEW_SERIAL}"
openssl x509 -in ${WORKDIR}/est/device1_re_leaf.pem -noout -subject -dates
openssl verify -CAfile ${WORKDIR}/est/ca_chain.pem ${WORKDIR}/est/device1_re_leaf.pem

if [ -z "${NEW_SERIAL}" ] || [ "${OLD_SERIAL}" = "${NEW_SERIAL}" ]; then
  echo "FAIL: re-enroll did not issue a new serial"
  exit 1
fi

CERT_PUB=$(openssl x509 -in ${WORKDIR}/est/device1_re_leaf.pem -noout -pubkey | openssl md5)
KEY_PUB=$(openssl rsa -in ${WORKDIR}/est/device1_re.key -pubout 2>/dev/null | openssl md5)
if [ "${CERT_PUB}" != "${KEY_PUB}" ]; then
  echo "FAIL: renewed certificate does not match device1_re.key"
  exit 1
fi
echo "OK: re-enroll issued a new serial bound to the new key"

cp ${WORKDIR}/est/device1_re_leaf.pem ${WORKDIR}/est/device1_leaf.pem
cp ${WORKDIR}/est/device1_re.key ${WORKDIR}/est/device1.key
echo "${NEW_SERIAL}" > ${WORKDIR}/est/device1.serial

## Step 7: Enrollment and re-enrollment with TLS client certificates

Bootstrap a factory credential with the admin token (`pki_est/issue/est-clients`). That cert authenticates subsequent EST calls via `curl --cert/--key` (no Basic header — otherwise userpass would win).

### Bootstrap a TLS client cert, then EST `simpleenroll`

In [ ]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"

echo "=== bootstrap leaf via PKI issue (admin token) ==="
vault write -format=json pki_est/issue/est-clients \
     common_name="bootstrap.est.example.com" \
     ttl="12h" \
     | tee ${WORKDIR}/est/bootstrap.json >/dev/null

jq -r '.data.certificate' ${WORKDIR}/est/bootstrap.json > ${WORKDIR}/est/bootstrap.pem
jq -r '.data.private_key' ${WORKDIR}/est/bootstrap.json > ${WORKDIR}/est/bootstrap.key
openssl x509 -in ${WORKDIR}/est/bootstrap.pem -noout -subject -serial

echo
echo "=== sanity: cert auth login with bootstrap cert (does not replace VAULT_TOKEN) ==="
curl -k -sS --cert ${WORKDIR}/est/bootstrap.pem --key ${WORKDIR}/est/bootstrap.key \
  --request POST --data '{"name":"est-clients"}' \
  ${VAULT_ADDR}/v1/auth/est-cert/login \
  | jq '{policies:.auth.policies, token_type:.auth.token_type, cert_name:.auth.metadata.cert_name, common_name:.auth.metadata.common_name}'

CN="tls-device.est.example.com"
openssl req -new -newkey rsa:2048 -nodes \
  -keyout ${WORKDIR}/est/tls_device.key \
  -out ${WORKDIR}/est/tls_device.csr \
  -subj "/CN=${CN}" 2>/dev/null
openssl req -in ${WORKDIR}/est/tls_device.csr -outform DER \
  | openssl base64 -e > ${WORKDIR}/est/tls_device.p10

echo
echo "=== POST /.well-known/est/simpleenroll (TLS client cert, no Basic) ==="
curl -k -sS -D ${WORKDIR}/est/enroll_tls.hdr \
  -o ${WORKDIR}/est/tls_device.p7 \
  --cert ${WORKDIR}/est/bootstrap.pem \
  --key ${WORKDIR}/est/bootstrap.key \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/tls_device.p10 \
  "${EST_BASE}/simpleenroll"

grep -iE 'HTTP/|content-type|content-transfer-encoding|www-authenticate' ${WORKDIR}/est/enroll_tls.hdr
HTTP_CODE=$(awk 'NR==1 {print $2}' ${WORKDIR}/est/enroll_tls.hdr)
if [ "${HTTP_CODE}" != "200" ]; then
  echo "EST TLS enroll failed HTTP ${HTTP_CODE}"
  cat ${WORKDIR}/est/tls_device.p7; echo
  exit 1
fi

${WORKDIR}/est/p7_to_pem.sh ${WORKDIR}/est/tls_device.p7 ${WORKDIR}/est/tls_device.pem
openssl x509 -in ${WORKDIR}/est/tls_device.pem -out ${WORKDIR}/est/tls_device_leaf.pem

echo
echo "=== issued TLS-enrolled certificate ==="
openssl x509 -in ${WORKDIR}/est/tls_device_leaf.pem -noout -subject -issuer -serial -dates
openssl verify -CAfile ${WORKDIR}/est/ca_chain.pem ${WORKDIR}/est/tls_device_leaf.pem
openssl x509 -in ${WORKDIR}/est/tls_device_leaf.pem -noout -serial | tee ${WORKDIR}/est/tls_device.serial

### TLS — `simplereenroll` using the just-enrolled device cert

Same as Basic reenroll, but the TLS client cert is also the Vault authenticator (`est-cert`): present the current leaf over TLS and send a rekey CSR whose Subject (`PrintableString`) and SAN match it.

In [ ]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"
CN="tls-device.est.example.com"
OLD_SERIAL=$(cat ${WORKDIR}/est/tls_device.serial)

cat > ${WORKDIR}/est/reenroll_req.cnf <<EOF
[req]
distinguished_name = dn
req_extensions = ext
string_mask = nombstr
prompt = no
utf8 = no
[dn]
CN = ${CN}
[ext]
subjectAltName = DNS:${CN}
EOF
openssl req -new -newkey rsa:2048 -nodes \
  -keyout ${WORKDIR}/est/tls_device_re.key \
  -out ${WORKDIR}/est/tls_device_re.csr \
  -config ${WORKDIR}/est/reenroll_req.cnf \
  -reqexts ext
openssl req -in ${WORKDIR}/est/tls_device_re.csr -noout -text | grep -A1 "Subject Alternative Name"
openssl req -in ${WORKDIR}/est/tls_device_re.csr -outform DER \
  | openssl base64 -e > ${WORKDIR}/est/tls_device_re.p10

echo "=== POST /.well-known/est/simplereenroll (TLS client cert of current leaf) ==="
curl -k -sS -D ${WORKDIR}/est/reenroll_tls.hdr \
  -o ${WORKDIR}/est/tls_device_re.p7 \
  --cert ${WORKDIR}/est/tls_device_leaf.pem \
  --key ${WORKDIR}/est/tls_device.key \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/tls_device_re.p10 \
  "${EST_BASE}/simplereenroll"

grep -iE 'HTTP/|content-type|content-transfer-encoding' ${WORKDIR}/est/reenroll_tls.hdr
HTTP_CODE=$(awk 'NR==1 {print $2}' ${WORKDIR}/est/reenroll_tls.hdr)
if [ "${HTTP_CODE}" != "200" ]; then
  echo "EST TLS reenroll failed HTTP ${HTTP_CODE}"
  cat ${WORKDIR}/est/tls_device_re.p7; echo
  exit 1
fi

${WORKDIR}/est/p7_to_pem.sh ${WORKDIR}/est/tls_device_re.p7 ${WORKDIR}/est/tls_device_re.pem
openssl x509 -in ${WORKDIR}/est/tls_device_re.pem -out ${WORKDIR}/est/tls_device_re_leaf.pem

NEW_SERIAL=$(openssl x509 -in ${WORKDIR}/est/tls_device_re_leaf.pem -noout -serial)
echo "old ${OLD_SERIAL}"
echo "new ${NEW_SERIAL}"
openssl x509 -in ${WORKDIR}/est/tls_device_re_leaf.pem -noout -subject -dates
openssl verify -CAfile ${WORKDIR}/est/ca_chain.pem ${WORKDIR}/est/tls_device_re_leaf.pem

if [ -z "${NEW_SERIAL}" ] || [ "${OLD_SERIAL}" = "${NEW_SERIAL}" ]; then
  echo "FAIL: TLS re-enroll did not issue a new serial"
  exit 1
fi

CERT_PUB=$(openssl x509 -in ${WORKDIR}/est/tls_device_re_leaf.pem -noout -pubkey | openssl md5)
KEY_PUB=$(openssl rsa -in ${WORKDIR}/est/tls_device_re.key -pubout 2>/dev/null | openssl md5)
if [ "${CERT_PUB}" != "${KEY_PUB}" ]; then
  echo "FAIL: renewed certificate does not match tls_device_re.key"
  exit 1
fi
echo "OK: TLS re-enroll issued a new serial bound to the new key"

cp ${WORKDIR}/est/tls_device_re_leaf.pem ${WORKDIR}/est/tls_device_leaf.pem
cp ${WORKDIR}/est/tls_device_re.key ${WORKDIR}/est/tls_device.key
echo "${NEW_SERIAL}" > ${WORKDIR}/est/tls_device.serial

## Step 8: Negative checks and issued inventory

Unauthenticated enroll must fail. Wrong Basic credentials must fail. Listing `pki_est/certs` should show the leaves created above.

In [ ]:
%%bash
EST_BASE="${VAULT_ADDR}/.well-known/est"

echo "=== enroll with no credentials (expect 401) ==="
curl -k -sS -o ${WORKDIR}/est/noauth.body -w "HTTP %{http_code}\n" \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/device1.p10 \
  "${EST_BASE}/simpleenroll"
echo "body:"
head -c 400 ${WORKDIR}/est/noauth.body; echo

echo
echo "=== enroll with wrong password (expect 4xx) ==="
curl -k -sS -o ${WORKDIR}/est/badpass.body -w "HTTP %{http_code}\n" \
  --user "estuser:wrongpass" \
  -H "Content-Type: application/pkcs10" \
  -H "Content-Transfer-Encoding: base64" \
  --data-binary @${WORKDIR}/est/device1.p10 \
  "${EST_BASE}/simpleenroll"
echo "body:"
head -c 400 ${WORKDIR}/est/badpass.body; echo

echo
echo "=== labeled EST path /.well-known/est/est-clients/cacerts ==="
curl -k -sS -o /dev/null -w "HTTP %{http_code}\n" \
  "${EST_BASE}/est-clients/cacerts"

echo
echo "=== certificates stored on pki_est ==="
curl -k --header "X-Vault-Token: $VAULT_TOKEN" --request LIST --silent \
  ${VAULT_ADDR}/v1/pki_est/certs | jq -r '.data.keys[]' | tee ${WORKDIR}/est/serials.txt

echo
echo "=== leaf CNs on pki_est ==="
while IFS= read -r SERIAL; do
  CN=$(curl -k --header "X-Vault-Token: $VAULT_TOKEN" --silent \
    ${VAULT_ADDR}/v1/pki_est/cert/${SERIAL} | jq -r '.data.certificate' \
    | openssl x509 -noout -subject -nameopt RFC2253 2>/dev/null)
  echo "${SERIAL}  ${CN}"
done < ${WORKDIR}/est/serials.txt

## Clean UP EST